# IEEE-CIS Notebook to Modular Code Migration

This notebook helps migrate logic from your original notebook into modular Python files and validates each stage.

## Migration Plan

1. Validate paths and data access
2. Re-run EDA from module
3. Re-run graph feature build from module
4. Re-run CatBoost training from module
5. Add agentic risk explanation on predictions
6. Compare notebook outputs vs module outputs

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError('Run this notebook from Dataset_IEEE_CIS root folder.')

# Optional for Colab users
# os.environ['FRAUD_DATA_ROOT'] = '/content/drive/MyDrive/Colab Notebooks/IEEE_CIS_Dataset'

print('Project root:', PROJECT_ROOT)

In [ ]:
%pip install -r requirements.txt
%pip install -e .

## Stage 1: Load and Join Data

In [ ]:
from fraud_detection.data.loader import load_raw_data, build_training_frame
from fraud_detection.settings import get_project_paths

paths = get_project_paths()
print('Data root:', paths.data_root)

bundle = load_raw_data(sample=True, nrows=50000)
train_df = build_training_frame(bundle)
print('Train shape:', train_df.shape)
train_df.head()

## Stage 2: EDA via Module

In [ ]:
from fraud_detection.features.eda import run_basic_eda

eda_artifacts = run_basic_eda(train_df)
print('Summary CSV:', eda_artifacts.summary_csv)
print('Missingness CSV:', eda_artifacts.missingness_csv)
print('Target Plot:', eda_artifacts.target_plot_png)

## Stage 3: Graph Build via Module

In [ ]:
from fraud_detection.models.graph_trainer import train_graph_features

graph_result = train_graph_features(train_df)
print('Graph file:', graph_result.graph_path)
print('Nodes:', graph_result.node_count)
print('Edges:', graph_result.edge_count)

## Stage 4: CatBoost Training via Module

In [ ]:
from fraud_detection.features.preprocessing import basic_feature_cleanup, split_xy
from fraud_detection.models.catboost_trainer import train_catboost

clean_df = basic_feature_cleanup(train_df)
x, y = split_xy(clean_df)
cat_result = train_catboost(x, y)
print('Model file:', cat_result.model_path)
print('Validation AUC:', round(cat_result.val_auc, 6))

## Stage 5: Agentic Risk Layer on Predictions

In [ ]:
from fraud_detection.agentic.risk_agent import explain_risk, to_dict
from fraud_detection.serving.predict import load_model, predict_proba

model = load_model(cat_result.model_path)
sample_x = x.head(10).copy()
scores = predict_proba(model, sample_x)

decisions = [to_dict(explain_risk(float(s))) for s in scores]
decisions[:3]

## Stage 6: Migration Tracker

Use this checklist while moving logic out of the original notebook:

- Data loading and joins migrated
- EDA tables and plots migrated
- Feature engineering migrated
- CatBoost model training migrated
- Graph/entity modeling migrated
- Inference + threshold logic migrated
- Streamlit and agentic explanation integrated

In [ ]:
print('Migration workflow notebook completed successfully if all stages run without errors.')